In [0]:
# Notebook 03 - Gold
# Description: Create Star Schema from Silver layer data

SILVER_PATH = "/Volumes/olist_project/silver/clean_data"

df_customers = spark.read.parquet(f"{SILVER_PATH}/customers")
df_orders = spark.read.parquet(f"{SILVER_PATH}/orders")
df_order_items = spark.read.parquet(f"{SILVER_PATH}/order_items")
df_payments = spark.read.parquet(f"{SILVER_PATH}/payments")
df_products = spark.read.parquet(f"{SILVER_PATH}/products")
df_sellers  = spark.read.parquet(f"{SILVER_PATH}/sellers")
df_category = spark.read.parquet(f"{SILVER_PATH}/category")

print("Silver data loaded!")

Silver data loaded!


In [0]:
# Create dim_customers

dim_customers = df_customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
)

print("dim_customers created:", dim_customers.count(), "rows")

dim_customers created: 99441 rows


In [0]:
# Create dim_products

dim_products = df_products.join(df_category, "product_category_name", how="left").select(
    "product_id",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)

print("dim_products created:", dim_products.count(), "rows")

dim_products created: 32951 rows


In [0]:
# Create dim_sellers

dim_sellers = df_sellers.select(
    "seller_id",
    "seller_city",
    "seller_state"
)

print("dim_sellers created:", dim_sellers.count(), "rows")

dim_sellers created: 3095 rows


In [0]:
# Create dim_dates

from pyspark.sql.functions import year, month, dayofmonth, dayofweek, date_format, col

dim_dates = df_orders.select(
    "order_purchase_timestamp"
).dropDuplicates().select(
    col("order_purchase_timestamp").alias("date"),
    year("order_purchase_timestamp").alias("year"),
    month("order_purchase_timestamp").alias("month"),
    dayofmonth("order_purchase_timestamp").alias("day"),
    dayofweek("order_purchase_timestamp").alias("day_of_week"),
    date_format("order_purchase_timestamp", "EEEE").alias("day_name"),
    date_format("order_purchase_timestamp", "MMMM").alias("month_name")
)

print("dim_dates created:", dim_dates.count(), "rows")

dim_dates created: 98875 rows


In [0]:
# Create fact_orders

fact_orders = df_orders.join(df_order_items, on="order_id", how="left") \
    .join(df_payments, on="order_id", how="left").select(
        "order_id",
        "customer_id",
        "seller_id",
        "product_id",
        "order_status",
        "order_purchase_timestamp",
        "payment_type",
        "payment_value",
        "price",
        "freight_value"
    )

print("fact_orders created:", fact_orders.count(), "rows")

fact_orders created: 118434 rows


In [0]:
# /Volumes/olist_project/gold/star_shemaSave Gold layer tables

GOLD_PATH = "/Volumes/olist_project/gold/star_schema"

dim_customers.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_customers")
dim_products.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_products")
dim_sellers.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_sellers")
dim_dates.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_dates")
fact_orders.write.mode("overwrite").parquet(f"{GOLD_PATH}/fact_orders")

print("Gold layer saved successfully!")

Gold layer saved successfully!
